In [1]:
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

# Fixa os resultados e escolhe GPU quando disponível.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_PATH = "HateBR/dataset/HateBR.csv"

# Carrega apenas o texto e o rótulo necessários para a classificação.
df = pd.read_csv(DATASET_PATH, encoding="utf-8")
df = df[["comentario", "label_final"]].dropna()
df["comentario"] = df["comentario"].astype(str).str.strip()
df["label_final"] = pd.to_numeric(df["label_final"], errors="coerce")
df = df.dropna()
df = df[df["comentario"] != ""]
df["label_final"] = df["label_final"].astype(int).clip(0, 1)
sentences = df["comentario"].tolist()
labels = df["label_final"].tolist()
# Mantém a proporção das classes ao separar treino e teste.
train_sentences, test_sentences, train_labels, test_labels = train_test_split(
    sentences, labels, test_size=0.2, random_state=SEED, stratify=labels
)
print(f"Registros: {len(sentences)} | treino: {len(train_sentences)} | teste: {len(test_sentences)}")
print(f"Discurso de odio: {sum(labels)} ({100 * np.mean(labels):.1f}%)")
print(f"Dispositivo: {DEVICE}")

Registros: 7000 | treino: 5600 | teste: 1400
Discurso de odio: 3500 (50.0%)
Dispositivo: cuda


In [2]:
# Converte cada comentário em tokens, preservando acentos e pontuação.
def tokenize(sentence):
    return re.findall(r"[\wÀ-ÿ]+|[^\w\s]", sentence.lower(), flags=re.UNICODE)

# 0 representa padding; 1 representa palavras ausentes no vocabulário.
PAD_TOKEN = 0
UNK_TOKEN = 1
vocab = {"<PAD>": PAD_TOKEN, "<UNK>": UNK_TOKEN}
# O vocabulário é criado somente com o treino para evitar vazamento de dados.
for sentence in train_sentences:
    for word in tokenize(sentence):
        if word not in vocab:
            vocab[word] = len(vocab)

def encode(sentence):
    return [vocab.get(word, UNK_TOKEN) for word in tokenize(sentence)] or [UNK_TOKEN]

train_sequences = [encode(sentence) for sentence in train_sentences]
test_sequences = [encode(sentence) for sentence in test_sentences]
max_len = min(80, max(len(sequence) for sequence in train_sequences + test_sequences))
# Converte as sequências para tensores e completa frases menores com padding.
def make_tensors(sequences, targets):
    lengths = torch.tensor([min(len(sequence), max_len) for sequence in sequences], dtype=torch.long)
    padded = [sequence[:max_len] + [PAD_TOKEN] * (max_len - len(sequence)) for sequence in sequences]
    return torch.tensor(padded, dtype=torch.long), lengths, torch.tensor(targets, dtype=torch.float32)

X_train, lengths_train, y_train = make_tensors(train_sequences, train_labels)
X_test, lengths_test, y_test = make_tensors(test_sequences, test_labels)
# Mini-lotes tornam o treinamento mais eficiente e embaralham os exemplos.
train_loader = DataLoader(TensorDataset(X_train, lengths_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, lengths_test, y_test), batch_size=256)
print(f"Vocabulario: {len(vocab)} tokens | tamanho maximo: {max_len}")

Vocabulario: 10676 tokens | tamanho maximo: 80


In [3]:
# Modelo: embedding, RNN e camada final para classificação binária.
class HateSpeechRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_TOKEN)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        # Ignora o padding durante a recorrência.
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.rnn(packed)
        return self.fc(self.dropout(hidden[-1])).squeeze(1)

model = HateSpeechRNN(len(vocab)).to(DEVICE)
# BCEWithLogitsLoss combina sigmoid e função de perda com estabilidade.
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

HateSpeechRNN(
  (embedding): Embedding(10676, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [4]:
# Mede perda e acurácia sem atualizar os pesos do modelo.
def evaluate(loader):
    model.eval()
    total_loss = correct = total = 0
    with torch.no_grad():
        for inputs, lengths, targets in loader:
            inputs, lengths, targets = inputs.to(DEVICE), lengths.to(DEVICE), targets.to(DEVICE)
            logits = model(inputs, lengths)
            total_loss += criterion(logits, targets).item() * len(targets)
            correct += ((torch.sigmoid(logits) >= 0.5) == targets.bool()).sum().item()
            total += len(targets)
    return total_loss / total, correct / total

# Treina por várias épocas e avalia o desempenho após cada uma.
epochs = 12
for epoch in range(epochs):
    model.train()
    for inputs, lengths, targets in train_loader:
        inputs, lengths, targets = inputs.to(DEVICE), lengths.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(inputs, lengths), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    test_loss, test_accuracy = evaluate(test_loader)
    print(f"Epoca {epoch + 1:02d}/{epochs} | loss teste: {test_loss:.4f} | acuracia: {test_accuracy:.3f}")

Epoca 01/12 | loss teste: 0.6379 | acuracia: 0.611
Epoca 02/12 | loss teste: 0.5853 | acuracia: 0.704
Epoca 03/12 | loss teste: 0.5535 | acuracia: 0.726
Epoca 04/12 | loss teste: 0.5904 | acuracia: 0.693
Epoca 05/12 | loss teste: 0.5889 | acuracia: 0.718
Epoca 06/12 | loss teste: 0.5822 | acuracia: 0.731
Epoca 07/12 | loss teste: 0.6630 | acuracia: 0.724
Epoca 08/12 | loss teste: 0.6430 | acuracia: 0.742
Epoca 09/12 | loss teste: 0.7674 | acuracia: 0.730
Epoca 10/12 | loss teste: 0.7829 | acuracia: 0.745
Epoca 11/12 | loss teste: 0.8349 | acuracia: 0.746
Epoca 12/12 | loss teste: 0.9348 | acuracia: 0.751


In [ ]:
# Aplica o mesmo pré-processamento do treino e retorna classe e probabilidade.
def predict(sentence):
    model.eval()
    sequence = encode(sentence)[:max_len]
    length = torch.tensor([len(sequence)], dtype=torch.long, device=DEVICE)
    inputs = torch.tensor([sequence + [PAD_TOKEN] * (max_len - len(sequence))], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        probability = torch.sigmoid(model(inputs, length)).item()
    label = "discurso de odio" if probability >= 0.5 else "nao e discurso de odio"
    return {"classe": label, "probabilidade": probability}

for phrase in ["Essa pessoa merece respeito.", "Voce e um lixo, seu idiota.", "A comunidade deve trabalhar unida."]:
    print(f"{phrase} -> {predict(phrase)}")

Essa pessoa merece respeito. -> {'classe': 'nao e discurso de odio', 'probabilidade': 0.025450486689805984}
Voce e um lixo, seu idiota. -> {'classe': 'discurso de odio', 'probabilidade': 0.9983277916908264}
A comunidade deve trabalhar unida. -> {'classe': 'nao e discurso de odio', 'probabilidade': 0.0027062587905675173}
te estuprei amor da minha vida -> {'classe': 'nao e discurso de odio', 'probabilidade': 0.000756516819819808}
